# IoU-F1 Evaluation with XGBoost
Retrains XGBoost (since PyTorch model collapsed to all-zeros)
Compare against StandUp4AI: IoU-F1=0.51 @ IoU=0.2

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os, json, numpy as np, pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = BASE + '/features_221'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'

print('Setup complete')


In [ ]:
# Load features + labels (timestamp-based)
feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
print('Feature files:', len(feat_files))

def parse_timestamp(ts_str):
    ts_str = str(ts_str).strip()
    try:
        parts = ts_str.strip('[]').split(',')
        return float(parts[0]), float(parts[1])
    except:
        return None, None

X_list, y_list, vids_list = [], [], []

for fi, f in enumerate(feat_files):
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)  # (n_chunks, 791)
    labels_df = pd.read_csv(label_path)
    n_chunks = len(feats)
    chunk_dur = 5.0
    
    word_times, word_labels = [], []
    for _, row in labels_df.iterrows():
        t0, t1 = parse_timestamp(row['timestamp'])
        if t0 is not None:
            word_times.append((t0, t1))
            word_labels.append(str(row['label']).strip())
    
    chunk_labels = []
    for i in range(n_chunks):
        c0, c1 = i * chunk_dur, (i+1) * chunk_dur
        is_laugh = any(
            wl in ['B', 'I', 'L'] and w0 < c1 and w1 > c0
            for (w0, w1), wl in zip(word_times, word_labels)
        )
        chunk_labels.append(1 if is_laugh else 0)
    
    X_list.append(feats)
    y_list.append(np.array(chunk_labels))
    vids_list.extend([vid] * n_chunks)

X = np.vstack(X_list)
y = np.concatenate(y_list)
groups = np.array(vids_list)

print('X:', X.shape, 'y:', y.shape, 'pos rate:', round(y.mean(), 3))
print('Unique videos:', len(set(vids_list)))


In [ ]:
# Train XGBoost (the model that worked in training notebook)
print('Training XGBoost...')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train on ALL data (for IoU evaluation)
neg = (y == 0).sum()
pos = (y == 1).sum()
scale = neg / max(pos, 1)
print('pos_rate:', round(y.mean(), 3), 'scale:', round(scale, 2))

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=1
)
model.fit(X_scaled, y)
print('XGBoost trained')

# Get predictions
probs_all = model.predict_proba(X_scaled)[:, 1]
print('Predictions done, shape:', probs_all.shape)
print('Prob stats: min=', round(probs_all.min(), 4), 'max=', round(probs_all.max(), 4), 'mean=', round(probs_all.mean(), 4))


In [ ]:
# IoU functions
def get_gt_segments(labels_df):
    segments = []
    i = 0
    while i < len(labels_df):
        lbl = str(labels_df.iloc[i]['label']).strip()
        t0, t1 = parse_timestamp(labels_df.iloc[i]['timestamp'])
        if t0 is None:
            i += 1
            continue
        if lbl == 'L':
            segments.append((t0, t1))
        elif lbl == 'B':
            start, end = t0, t1
            j = i + 1
            while j < len(labels_df):
                nl = str(labels_df.iloc[j]['label']).strip()
                if nl in ('I', 'L'):
                    _, end = parse_timestamp(labels_df.iloc[j]['timestamp'])
                    j += 1
                else:
                    break
            segments.append((start, end))
            i = j - 1
        i += 1
    return segments

def span_iou(s1, s2):
    inter = max(0, min(s1[1], s2[1]) - max(s1[0], s2[0]))
    union = max(s1[1], s2[1]) - min(s1[0], s2[0])
    return inter / union if union > 0 else 0

def chunk_preds_to_segments(probs, timestamps, threshold=0.5):
    segments = []
    in_seg = False
    seg_start = 0
    for i, (p, (t0, t1)) in enumerate(zip(probs, timestamps)):
        if p >= threshold and not in_seg:
            in_seg = True
            seg_start = t0
        elif p < threshold and in_seg:
            in_seg = False
            segments.append((seg_start, t0))
    if in_seg:
        segments.append((seg_start, timestamps[-1][1]))
    return segments

def segment_f1(pred_segs, gt_segs, iou_thresh=0.2):
    if not pred_segs or not gt_segs:
        return 0.0, 0.0, 0.0
    matched_pred, matched_gt = set(), set()
    for pi, ps in enumerate(pred_segs):
        best_iou, best_gi = 0.0, -1
        for gi, gs in enumerate(gt_segs):
            if gi in matched_gt:
                continue
            iou_val = span_iou(ps, gs)
            if iou_val >= iou_thresh and iou_val > best_iou:
                best_iou, best_gi = iou_val, gi
        if best_gi >= 0:
            matched_pred.add(pi)
            matched_gt.add(best_gi)
    tp = len(matched_pred)
    p = tp / len(pred_segs) if pred_segs else 0.0
    r = tp / len(gt_segs) if gt_segs else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f

print('IoU functions defined')


In [ ]:
# Evaluate at multiple IoU thresholds
IOU_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])

idx = 0
results_by_iou = {th: [] for th in IOU_THRESHOLDS}
per_video = []

for f in feat_files:
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)
    labels_df = pd.read_csv(label_path)
    n_chunks = len(feats)
    chunk_dur = 5.0
    
    probs = probs_all[idx:idx+n_chunks]
    idx += n_chunks
    
    timestamps = [(i * chunk_dur, (i+1) * chunk_dur) for i in range(n_chunks)]
    
    # Sweep threshold to find best
    best_th, best_f = 0.5, 0.0
    for th in [0.3, 0.4, 0.5, 0.6, 0.7]:
        pred_segs = chunk_preds_to_segments(probs, timestamps, threshold=th)
        gt_segs = get_gt_segments(labels_df)
        _, _, f = segment_f1(pred_segs, gt_segs, iou_thresh=0.2)
        if f > best_f:
            best_f = f
            best_th = th
    
    pred_segs = chunk_preds_to_segments(probs, timestamps, threshold=best_th)
    gt_segs = get_gt_segments(labels_df)
    
    row = {'vid': vid, 'n_pred': len(pred_segs), 'n_gt': len(gt_segs), 'best_th': best_th}
    for th in IOU_THRESHOLDS:
        p, r, f = segment_f1(pred_segs, gt_segs, th)
        row['p_' + str(th)] = round(p, 4)
        row['r_' + str(th)] = round(r, 4)
        row['f_' + str(th)] = round(f, 4)
        results_by_iou[th].append({'vid': vid, 'p': p, 'r': r, 'f': f})
    per_video.append(row)

print('Evaluated', len(per_video), 'videos')
n_pred_all = sum(r['n_pred'] for r in per_video)
n_gt_all = sum(r['n_gt'] for r in per_video)
print('Total pred segments:', n_pred_all, '| Total gt segments:', n_gt_all)


In [ ]:
# Results summary
print('=' * 60)
print('IoU-F1 EVALUATION RESULTS')
print('=' * 60)
print(f"{'IoU':>6} | {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 45)

summary = {}
for th in IOU_THRESHOLDS:
    rs = results_by_iou[th]
    if not rs:
        continue
    pm = np.mean([x['p'] for x in rs])
    rm = np.mean([x['r'] for x in rs])
    fm = np.mean([x['f'] for x in rs])
    summary[th] = {'p': pm, 'r': rm, 'f': fm}
    print(f" >= {th:.1f} | {pm:.4f} {rm:.4f} {fm:.4f}")

print('')
print('StandUp4AI (EMNLP 2025): IoU-F1=0.51 @ IoU=0.2')
print('')

best_iou = max(summary.keys(), key=lambda th: summary[th]['f'])
print(f"Best: IoU >= {best_iou}, F1={summary[best_iou]['f']:.4f}")

print('')
print('Top 10 by IoU-F1@0.2:')
print(f"{'Video':<20} {'n_pred':>6} {'n_gt':>5} {'P':>8} {'R':>8} {'F1':>8}")
print('-' * 60)
for row in sorted(per_video, key=lambda x: x.get('f_0.2', 0), reverse=True)[:10]:
    print(f"{row['vid']:<20} {row['n_pred']:>6} {row['n_gt']:>5} "
          f"{row.get('p_0.2', 0):>8.4f} {row.get('r_0.2', 0):>8.4f} {row.get('f_0.2', 0):>8.4f}")

out = {'iou_thresholds': IOU_THRESHOLDS, 'summary': summary, 'per_video': per_video}
with open(BASE + '/iou_f1_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print('')
print('Saved to', BASE + '/iou_f1_results.json')
